# California Housing Value Prediction

Compare linear models with original and engineered features on historical census block-group data. This educational Datheon task demonstrates data preparation, five-fold validation, regularisation, regression metrics, and error analysis. It does not estimate current individual property prices.

**Run:** In Colab select **Runtime → Run all**. Locally follow the [project README](README.md). The runner uses the same notebook cells. Random seeds are fixed at 42.

<a href="https://colab.research.google.com/github/primrosemandy93-maker/Datheon_Machine_Learning_-_Data_Science_Internship/blob/main/task_2_housing/Task_2_Housing_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Tools and reproducible output paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Tools imported successfully.")

# Supports execution from the repository root or this project folder.
PROJECT_DIR = next((p / "task_2_housing" for p in [Path.cwd(), *Path.cwd().parents] if (p / "task_2_housing").is_dir()), Path.cwd())
OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Tools imported successfully.


## 2. Load and verify the exact dataset

In [2]:
# The original notebook used Colab's 17,000-row training CSV.
# Do not substitute sklearn's 20,640-row California dataset.
import hashlib
import os
from urllib.request import urlopen

DATA_URL = "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"
DATA_SHA256 = "bb9ca4041a66e0ed691cb825188f16fdab518435c4950a080951be2fdfe30d4d"
override = os.environ.get("HOUSING_CSV")
candidates = [Path(override)] if override else [
    PROJECT_DIR / "data" / "california_housing_train.csv",
    Path("/content/sample_data/california_housing_train.csv"),
]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    if override:
        raise FileNotFoundError(f"HOUSING_CSV does not exist: {override}")
    data_path = PROJECT_DIR / "data" / "california_housing_train.csv"
    data_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            data_bytes = response.read()
    except Exception as exc:
        raise RuntimeError("Dataset download failed. Download the CSV from DATA_URL and set HOUSING_CSV to its local path; see README.md.") from exc
    if hashlib.sha256(data_bytes).hexdigest() != DATA_SHA256:
        raise ValueError("Downloaded dataset checksum differs; refusing to report incomparable results.")
    data_path.write_bytes(data_bytes)
data_bytes = data_path.read_bytes()
if hashlib.sha256(data_bytes).hexdigest() != DATA_SHA256:
    raise ValueError("Dataset checksum differs from the original 17,000-row CSV. Use the exact file documented in README.md.")
housing = pd.read_csv(data_path)
assert housing.shape == (17000, 9), "Unexpected housing dataset shape"
print("Verified dataset SHA-256:", DATA_SHA256)
print("Dataset shape:", housing.shape)
display(housing.head())
print("\nMissing values:")
print(housing.isna().sum())


Verified dataset SHA-256: bb9ca4041a66e0ed691cb825188f16fdab518435c4950a080951be2fdfe30d4d
Dataset shape: (17000, 9)

Missing values:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


## 3. Reserve the test set

In [3]:
# Features: information used to predict house values.
X = housing.drop(columns=["median_house_value"])

# Target: the house value we want to predict.
y = housing["median_house_value"]

# Reserve 20% for final testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nFeatures:")
print(X.columns.tolist())

Training rows: 13600
Testing rows: 3400

Features:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']


## 4. Explore training data only

In [4]:
# Explore training data only, keeping the test set separate.
training_data = X_train.copy()
training_data["median_house_value"] = y_train

plt.figure(figsize=(11, 8))

sns.heatmap(
    training_data.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
)

plt.title("Correlations in the housing training data")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_correlations.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/training_correlations.png)

## 5. Engineer ratios and an interaction

In [5]:
def add_housing_features(data):
    data = data.copy()

    households = data["households"].replace(0, np.nan)
    rooms = data["total_rooms"].replace(0, np.nan)

    data["rooms_per_household"] = data["total_rooms"] / households
    data["bedrooms_per_room"] = data["total_bedrooms"] / rooms
    data["population_per_household"] = data["population"] / households

    data["income_x_rooms_per_household"] = (
        data["median_income"] * data["rooms_per_household"]
    )

    return data.replace([np.inf, -np.inf], np.nan)


X_train_engineered = add_housing_features(X_train)
X_test_engineered = add_housing_features(X_test)

print("Engineered features created successfully.")

Engineered features created successfully.


## 6. Compare original and engineered features with cross-validation

In [6]:
# Keep preprocessing inside each pipeline to prevent data leakage.
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

engineered_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

# Use the same validation splits for a fair comparison.
cv = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.model_selection import cross_val_score

comparisons = [
    ("Original features", baseline_model, X_train),
    ("Engineered features", engineered_model, X_train_engineered),
]

comparison_results = []

for name, model, features in comparisons:
    scores = cross_val_score(
        model,
        features,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
    )

    rmse_scores = -scores

    comparison_results.append({
        "Features": name,
        "Mean validation RMSE ($)": rmse_scores.mean(),
        "RMSE standard deviation ($)": rmse_scores.std(),
    })

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df.round(2))

,Features,Mean validation RMSE ($),RMSE standard deviation ($)
0,Original features,70049.11,1859.51
1,Engineered features,70805.39,5178.29


## 7. Tune regularisation within training data

In [7]:
# Try several regularization strengths.
alpha_values = [0.01, 0.1, 1, 10, 100]

tuned_models = {}

for name, estimator in {
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=50000),
}.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid={"model__alpha": alpha_values},
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=1,
        error_score="raise",
    )

    search.fit(X_train_engineered, y_train)
    tuned_models[name] = search.best_estimator_

    print(f"\n{name}")
    print("Selected alpha:", search.best_params_["model__alpha"])
    print("Mean validation RMSE ($):",
          round(-search.best_score_, 2))


Ridge
Selected alpha: 0.01
Mean validation RMSE ($): 70805.52

Lasso
Selected alpha: 0.01
Mean validation RMSE ($): 70805.42


## 8. Report held-out test results

In [8]:
# Train both Linear Regression versions on all training rows.
baseline_model.fit(X_train, y_train)
engineered_model.fit(X_train_engineered, y_train)

final_models = {
    "Linear Regression — original": (baseline_model, X_test),
    "Linear Regression — engineered": (
        engineered_model, X_test_engineered
    ),
    "Ridge — engineered": (
        tuned_models["Ridge"], X_test_engineered
    ),
    "Lasso — engineered": (
        tuned_models["Lasso"], X_test_engineered
    ),
}

test_results = []
test_predictions = {}

for name, (model, features) in final_models.items():
    predictions = model.predict(features)
    test_predictions[name] = predictions

    test_results.append({
        "Model": name,
        "RMSE ($)": np.sqrt(
            mean_squared_error(y_test, predictions)
        ),
        "MAE ($)": mean_absolute_error(y_test, predictions),
        "R²": r2_score(y_test, predictions),
    })

test_results_df = pd.DataFrame(test_results)
display(test_results_df.round(3))

,Model,RMSE ($),MAE ($),R²
0,Linear Regression — original,68078.326,49983.475,0.664
1,Linear Regression — engineered,67358.059,49219.454,0.671
2,Ridge — engineered,67358.064,49219.444,0.671
3,Lasso — engineered,67358.062,49219.453,0.671


## 9. Inspect actual versus predicted values

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, predictions) in zip(
    axes.flat, test_predictions.items()
):
    ax.scatter(
        y_test,
        predictions,
        alpha=0.25,
        s=10,
    )

    lower = min(y_test.min(), predictions.min())
    upper = max(y_test.max(), predictions.max())

    # Perfect predictions would fall on this line.
    ax.plot(
        [lower, upper],
        [lower, upper],
        color="red",
        linestyle="--",
    )

    ax.set_title(name)
    ax.set_xlabel("Actual median house value ($)")
    ax.set_ylabel("Predicted median house value ($)")
    ax.ticklabel_format(style="plain", axis="both")

plt.suptitle("Actual versus predicted housing values")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/actual_vs_predicted.png)

## 10. Examine residuals

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (name, predictions) in zip(
    axes.flat, test_predictions.items()
):
    residuals = y_test.to_numpy() - predictions

    sns.histplot(
        residuals,
        bins=50,
        kde=True,
        ax=ax,
    )

    ax.axvline(0, color="red", linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Residual: actual − predicted value ($)")
    ax.set_ylabel("Number of test areas")

plt.suptitle("Distribution of prediction errors")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "residuals.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/residuals.png)

## 11. Compare errors by observed price band

In [11]:
# Group test areas by their actual median house value.
price_bands = pd.cut(
    y_test,
    bins=[-np.inf, 150000, 300000, np.inf],
    labels=["Under $150k", "$150k–$300k", "Above $300k"],
    right=False,
)

band_results = []

for name, predictions in test_predictions.items():
    errors = pd.DataFrame({
        "Price band": price_bands.to_numpy(),
        "Absolute error": np.abs(
            y_test.to_numpy() - predictions
        ),
    })

    summary = (
        errors.groupby("Price band", observed=True)
        .agg(
            MAE=("Absolute error", "mean"),
            Areas=("Absolute error", "size"),
        )
        .reset_index()
    )

    summary["Model"] = name
    band_results.append(summary)

band_results_df = pd.concat(band_results, ignore_index=True)

display(
    band_results_df[
        ["Model", "Price band", "Areas", "MAE"]
    ].round(2)
)

,Model,Price band,Areas,MAE
0,Linear Regression — original,$150k–$300k,1474,36353.96
1,Linear Regression — original,Above $300k,674,91016.76
2,Linear Regression — original,Under $150k,1252,43939.92
3,Linear Regression — engineered,$150k–$300k,1474,36179.36
4,Linear Regression — engineered,Above $300k,674,89805.01
5,Linear Regression — engineered,Under $150k,1252,42722.99
6,Ridge — engineered,$150k–$300k,1474,36179.25
7,Ridge — engineered,Above $300k,674,89805.23
8,Ridge — engineered,Under $150k,1252,42722.98
9,Lasso — engineered,$150k–$300k,1474,36179.33


## 12. Export measured results

In [12]:
print("OVERALL TEST RESULTS")
print(test_results_df.round(3).to_string(index=False))

print("\nERRORS BY PRICE BAND")
print(band_results_df.round(2).to_string(index=False))
test_results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "cross_validation.csv", index=False)
band_results_df.to_csv(OUTPUT_DIR / "errors_by_price_band.csv", index=False)


OVERALL TEST RESULTS
                         Model  RMSE ($)   MAE ($)    R²
  Linear Regression — original 68078.326 49983.475 0.664
Linear Regression — engineered 67358.059 49219.454 0.671
            Ridge — engineered 67358.064 49219.444 0.671
            Lasso — engineered 67358.062 49219.453 0.671

ERRORS BY PRICE BAND
 Price band      MAE  Areas                          Model
$150k–$300k 36353.96   1474   Linear Regression — original
Above $300k 91016.76    674   Linear Regression — original
Under $150k 43939.92   1252   Linear Regression — original
$150k–$300k 36179.36   1474 Linear Regression — engineered
Above $300k 89805.01    674 Linear Regression — engineered
Under $150k 42722.99   1252 Linear Regression — engineered
$150k–$300k 36179.25   1474             Ridge — engineered
Above $300k 89805.23    674             Ridge — engineered
Under $150k 42722.98   1252             Ridge — engineered
$150k–$300k 36179.33   1474             Lasso — engineered
Above $300k 89805.06   

## Results and Conclusion

### Did feature engineering improve predictions?

On the held-out test split, adding rooms per household, bedrooms per room, population per household, and an income–rooms interaction modestly improved Linear Regression performance.

However, five-fold validation on the training data was worse with engineered features: mean RMSE increased from $70,049.11 to $70,805.39, and the fold standard deviation increased from $1,859.51 to $5,178.29. The evidence is mixed: a better test result does not establish a robust general improvement. Prefer the original-feature model on the validation evidence; use further validation before changing that choice.

Test RMSE decreased from $68,078.33 to $67,358.06, approximately
a 1.1% reduction. MAE decreased from $49,983.48 to $49,219.45.
R² increased from 0.664 to 0.671.

These results evaluate the added features together; they do not
show which individual feature contributed the improvement.

### Did regularization help?

Ridge and Lasso produced almost identical test results to
Linear Regression with engineered features. With the settings
selected by cross-validation, regularization provided no
meaningful additional improvement on this test set.

### Where did predictions struggle?

For engineered Linear Regression, average absolute errors were:

- Under $150,000: $42,722.99 across 1,252 areas.
- $150,000 to below $300,000: $36,179.36 across 1,474 areas.
- $300,000 and above: $89,805.01 across 674 areas.

The highest price band had the largest dollar errors.
However, absolute dollar errors alone do not establish which
group had the largest percentage errors.

### Limitations

The data describes historical California census block groups,
rather than current individual property prices. The dataset
also caps the highest recorded house values.

A random split can place nearby areas in both training and
testing sets. Therefore, these results may not represent
performance in a geographically new region.

### Summary

Feature engineering provided a small test-set improvement but worse cross-validation performance. The three
models using engineered features performed similarly, and
higher-priced areas remained challenging to predict accurately.